# Day 18: GPU Generations — Hopper, Ada, Blackwell, Rubin
> *Inference Engineering* — Chapter 3.2 | Philip Kiely, Baseten Books 2026

**Layer:** Infrastructure | **Prerequisite:** Day 17 (GPU Architecture)


## Concept Overview

NVIDIA's GPU generations follow a predictable scaling trajectory. Ampere (A100) → Hopper (H100) brought FP8 Tensor Cores and NVLink 4.0. Ada Lovelace (RTX 40xx, L40S) added Ada Tensor Cores and higher FP8 throughput for edge/cloud inference. Blackwell (B100, B200, GB200) doubles Hopper's FP8 throughput and introduces FP4 precision, GB200 NVL72 racks, and a new interconnect architecture. Rubin follows Blackwell. Each generation roughly doubles the inference-relevant compute: 2x the FP8 TFLOP/s and 2x the memory bandwidth.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch

print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')


## 1. Generation Comparison

Comparing key inference-relevant specs across GPU generations.


In [ ]:
gpu_gens = [
    # (name, year, fp16_tflops, fp8_tflops, fp4_tflops, hbm_gb, hbm_bw_tbs, nvlink_bw_gbs, process_nm)
    ('A100 SXM4',  2020, 312,   None,  None, 80,  2.0,  600,  7),
    ('H100 SXM5',  2022, 989,   1979,  None, 80,  3.35, 900,  4),
    ('H200 SXM',   2023, 989,   1979,  None, 141, 4.8,  900,  4),
    ('L40S',       2023, 362,   733,   None, 48,  0.864,None, 4),
    ('B100 SXM',   2024, 1800,  3500,  7000, 192, 8.0,  1800, 4),
    ('B200 SXM',   2024, 2250,  4500,  9000, 192, 8.0,  1800, 4),
    ('GB10 (Spark)',2025, 67,   134,   None, 128, 0.273,None, 4),
]

print(f'{"GPU":<15} {"Year":>6} {"FP16":>8} {"FP8":>8} {"FP4":>8} {"HBM":>6} {"BW":>6} {"Node":>6}')
print(f'{"":15} {"":6} {"TFLOPS":>8} {"TFLOPS":>8} {"TFLOPS":>8} {"(GB)":>6} {"(TB/s)":>6} {"(nm)":>6}')
print('-' * 70)
for row in gpu_gens:
    name, year, fp16, fp8, fp4, hbm, bw, _, nm = row
    fp8_s = f'{fp8:.0f}' if fp8 else 'N/A'
    fp4_s = f'{fp4:.0f}' if fp4 else 'N/A'
    print(f'{name:<15} {year:>6} {fp16:>8.0f} {fp8_s:>8} {fp4_s:>8} {hbm:>6} {bw:>6.2f} {nm:>6}')


## 2. Generation-Over-Generation Scaling

Each major generation roughly doubles FP8 throughput — plotting the scaling trajectory.


In [ ]:
gens = [
    ('A100', 2020, 312),
    ('H100', 2022, 989*2),   # FP8 ≈ 2x FP16
    ('B200', 2024, 4500),    # FP8
]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
years = [g[1] for g in gens]
tflops = [g[2] for g in gens]
names = [g[0] for g in gens]
ax.semilogy(years, tflops, 'b-o', markersize=10)
for y, t, n in zip(years, tflops, names):
    ax.annotate(f'{n}\n{t:.0f}T', (y, t), textcoords='offset points', xytext=(5,5))
ax.set_xlabel('Year')
ax.set_ylabel('FP8 TFLOP/s')
ax.set_title('NVIDIA GPU FP8 Throughput by Generation')
ax.grid(True, which='both')

# Bandwidth scaling
ax2 = axes[1]
bw_data = [('A100', 2020, 2.0), ('H100', 2022, 3.35), ('H200', 2023, 4.8), ('B100', 2024, 8.0)]
yrs = [d[1] for d in bw_data]
bws = [d[2] for d in bw_data]
nms = [d[0] for d in bw_data]
ax2.plot(yrs, bws, 'r-o', markersize=10)
for y, b, n in zip(yrs, bws, nms):
    ax2.annotate(f'{n}\n{b}TB/s', (y, b), textcoords='offset points', xytext=(5,5))
ax2.set_xlabel('Year')
ax2.set_ylabel('HBM Bandwidth (TB/s)')
ax2.set_title('HBM Bandwidth Scaling')
ax2.grid(True)

plt.tight_layout()
plt.savefig('gpu_generations.png', dpi=100, bbox_inches='tight')
plt.show()
print('Saved gpu_generations.png')


## 3. New Features by Generation

Each generation adds capabilities beyond raw throughput that affect inference architecture decisions.


In [ ]:
features = {
    'A100 (Ampere)': [
        'FP16/BF16 Tensor Cores (3rd gen)',
        'A100 Multi-Instance GPU (MIG) — up to 7 instances',
        'NVLink 3.0: 600 GB/s bidirectional',
        'HBM2e: 2.0 TB/s',
        'sparsity acceleration (2:4 structured)',
    ],
    'H100 (Hopper)': [
        'FP8 Tensor Cores (first generation)',
        'Transformer Engine: auto FP8/FP16 selection',
        'NVLink 4.0: 900 GB/s',
        'HBM3: 3.35 TB/s',
        'In-order Tensor Core pipeline',
        'Confidential Computing support',
    ],
    'B200 (Blackwell)': [
        'FP4 Tensor Cores (first generation)',
        '2nd gen Transformer Engine',
        'NVLink 5.0: 1800 GB/s',
        'HBM3e: 8.0 TB/s',
        'RAS (Reliability/Availability/Serviceability) engine',
        'Decompression engine for database workloads',
    ],
}
for gen, feats in features.items():
    print(f'{gen}:')
    for f in feats:
        print(f'  + {f}')
    print()


## 4. Choosing Hardware for Inference

The right GPU depends on model size, latency requirements, and cost per token.


In [ ]:
# Cost-efficiency model: tokens per dollar
cost_data = [
    # (GPU, cost_hr_cloud, fp8_tflops, hbm_gb, hbm_bw_tbs)
    ('A100 80GB', 3.5,  312*2,  80,  2.0),
    ('H100 80GB', 10.0, 1979,   80,  3.35),
    ('H200',      12.0, 1979,   141, 4.8),
    ('L40S',      4.5,  733,    48,  0.864),
    ('B200',      20.0, 4500,   192, 8.0),
]

# For decode-bound workload: throughput ~ min(compute, BW)
# Decode throughput ∝ HBM BW (memory-bound)
# Prefill throughput ∝ FP8 TFLOPS (compute-bound)
print(f'{"GPU":<15} {"$/hr":>6} {"Decode (BW-bound)":>20} {"Prefill (compute)":>20}')
print(f'{"":<15} {"":>6} {"BW/cost":>20} {"TFLOPS/cost":>20}')
print('-' * 64)
for name, cost, tflops, hbm, bw in cost_data:
    print(f'{name:<15} {cost:>6.1f} {bw/cost:>20.2f} TB/s/$ {tflops/cost:>16.0f} TF/$')


## Experiments: Try These

1. **FP8 vs FP16 benchmark**: Run a GEMM benchmark with FP8 and FP16 on H100 (if available). Measure the actual throughput ratio vs the theoretical 2x.
2. **NVLink vs PCIe**: Benchmark point-to-point bandwidth on your DGX Spark pair using `torch.distributed`. How close to spec is the measured rate?
3. **MIG partitioning**: Configure MIG on a Spark GPU with `nvidia-smi mig` commands. Profile inference latency and throughput at 1g.10gb vs 2g.20gb vs 7g.80gb.


## Key Takeaways

- Each NVIDIA GPU generation roughly doubles FP8 throughput and HBM bandwidth every 2 years.
- H100 introduced FP8 Tensor Cores and the Transformer Engine — the first GPU architecturally optimized for LLM inference.
- Blackwell adds FP4, doubles H100's compute/bandwidth, and GB200 NVL72 racks enable 72-GPU NVLink domains for giant model serving.
- For memory-bound decode workloads, HBM bandwidth per dollar is the key metric — not raw TFLOP/s.

## References
- *Inference Engineering* Ch 3.2 — Philip Kiely, Baseten Books 2026
- NVIDIA Hopper Architecture White Paper
- NVIDIA Blackwell Architecture Technical Brief
